# 03 — Final Comparison

**This notebook does not train anything.** It reads only the CSV artefacts
produced by notebook 02 and builds the comparison tables, graphs and discussion.

### Hard rule on numbers
Every "Our ..." value is read from `results/metrics/*.csv`, which is written only
by code that actually executed on the GPU. Nothing is hard-coded here. The paper's
values are labelled as paper reference values and are **never** presented as our
experimental results.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

CLASSES = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]
MODEL_ORDER = ["VGG16", "ResNet18", "ResNet34", "D_ResNet"]


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p] + list(p.parents):
        if (cand / "notebooks").is_dir() and (cand / "results").is_dir():
            return cand
    return p


PROJECT_ROOT = find_project_root()
RESULTS = PROJECT_ROOT / "results"
for sub in ["comparisons", "metrics", "confusion_matrices", "training_curves"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

def R(sub):
    return RESULTS / sub

FINAL_CSV = R("metrics") / "final_results.csv"
if not FINAL_CSV.exists():
    raise SystemExit(
        f"Missing {FINAL_CSV}\nRun 02_Model_Experiments.ipynb first."
    )

print("PROJECT_ROOT :", PROJECT_ROOT)
print("reading      :", FINAL_CSV.relative_to(PROJECT_ROOT))

PROJECT_ROOT : /content/BSM-Project
reading      : results/metrics/final_results.csv


---
## 1. Model comparison table

In [2]:
final_results = pd.read_csv(FINAL_CSV)
final_results = final_results.set_index("model").reindex(
    [m for m in MODEL_ORDER if m in set(final_results["model"])]).reset_index()

comparison = pd.DataFrame({
    "Model": final_results["model"],
    "Architecture (torchvision)": final_results["architecture"],
    "Test Accuracy": final_results["test_accuracy"],
    "Test Precision (macro)": final_results["test_precision_macro"],
    "Test Recall (macro)": final_results["test_recall_macro"],
    "Test F1 (macro)": final_results["test_f1_macro"],
    "Test Specificity (macro)": final_results["test_specificity_macro"],
    "Test F1 (weighted)": final_results["test_f1_weighted"],
    "Best Epoch": final_results["best_epoch"],
    "Best Val Accuracy": final_results["best_val_accuracy"],
    "Training Time (min)": final_results["total_training_seconds"] / 60.0,
})
comparison.to_csv(R("comparisons") / "model_comparison.csv", index=False)
print(comparison.round(4).to_string(index=False))

   Model Architecture (torchvision)  Test Accuracy  Test Precision (macro)  Test Recall (macro)  Test F1 (macro)  Test Specificity (macro)  Test F1 (weighted)  Best Epoch  Best Val Accuracy  Training Time (min)
   VGG16                      vgg16         0.7679                  0.7725               0.7667           0.7667                    0.9536              0.7679           4             0.8182              26.2049
ResNet18                   resnet18         0.8036                  0.8093               0.8019           0.8018                    0.9608              0.8036          30             0.8727               7.0764
ResNet34                   resnet34         0.7321                  0.7314               0.7333           0.7303                    0.9464              0.7297           2             0.8364               9.2231
D_ResNet                   resnet50         0.6964                  0.7043               0.6944           0.6964                    0.9393              0.69

---
## 2. Model comparison graph

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
x = np.arange(len(comparison))
labels = comparison["Model"].tolist()

vals = comparison["Test Accuracy"].values
axes[0].bar(x, vals, color="#4C72B0")
axes[0].set_ylim(0, 1.05)
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=15, ha="right")
axes[0].set_ylabel("test accuracy"); axes[0].set_title("Test accuracy (our results)")
axes[0].grid(alpha=0.3, axis="y")

width = 0.2
metrics_specs = [("Test Precision (macro)", "#55A868"),
                 ("Test Recall (macro)", "#C44E52"),
                 ("Test F1 (macro)", "#8172B2"),
                 ("Test Specificity (macro)", "#937860")]
for i, (col, color) in enumerate(metrics_specs):
    axes[1].bar(x + (i - 1.5) * width, comparison[col].values, width, label=col.split(" (")[0], color=color)
axes[1].set_ylim(0, 1.05)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=15, ha="right")
axes[1].set_ylabel("score (macro avg)"); axes[1].set_title("Precision / Recall / F1 / Specificity")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis="y")

fig.suptitle("BSM mineral classification — our experimental results", fontsize=13)
plt.tight_layout(); plt.savefig(R("comparisons") / "model_comparison.png", dpi=150); plt.close()
print("  wrote results/comparisons/model_comparison.png")

  wrote results/comparisons/model_comparison.png


---
## 3. Paper vs our results

Paper values are transcribed from the publication and are used **only** as a
reference target. Our values are read from `results/metrics/final_results.csv`,
which only exists if training actually ran.

In [4]:
PAPER_REFERENCE = {
    "VGG16":    {"accuracy": 0.83, "precision": 0.89, "recall": 0.83},
    "ResNet18": {"accuracy": 0.78, "precision": 0.87, "recall": 0.78},
    "ResNet34": {"accuracy": 0.83, "precision": 0.89, "recall": 0.83},
    "D_ResNet": {"accuracy": 0.89, "precision": 0.90, "recall": 0.89},
}

ours = comparison.set_index("Model")
rows = []
for model in comparison["Model"]:
    p = PAPER_REFERENCE.get(model)
    if p is None:
        continue
    our_acc = float(ours.loc[model, "Test Accuracy"])
    our_prec = float(ours.loc[model, "Test Precision (macro)"])
    our_rec = float(ours.loc[model, "Test Recall (macro)"])
    rows.append({
        "Model": model,
        "Paper Accuracy": p["accuracy"],
        "Our Accuracy": our_acc,
        "Delta Accuracy": our_acc - p["accuracy"],
        "Paper Precision": p["precision"],
        "Our Precision (macro)": our_prec,
        "Delta Precision": our_prec - p["precision"],
        "Paper Recall": p["recall"],
        "Our Recall (macro)": our_rec,
        "Delta Recall": our_rec - p["recall"],
    })

paper_vs_ours = pd.DataFrame(rows)
paper_vs_ours.to_csv(R("comparisons") / "paper_vs_our_results.csv", index=False)
print(paper_vs_ours.round(4).to_string(index=False))
print("\n  'Paper ...' columns are published reference values.")
print("  'Our ...' columns come from executed training runs.")

   Model  Paper Accuracy  Our Accuracy  Delta Accuracy  Paper Precision  Our Precision (macro)  Delta Precision  Paper Recall  Our Recall (macro)  Delta Recall
   VGG16            0.83        0.7679         -0.0621             0.89                 0.7725          -0.1175          0.83              0.7667       -0.0633
ResNet18            0.78        0.8036          0.0236             0.87                 0.8093          -0.0607          0.78              0.8019        0.0219
ResNet34            0.83        0.7321         -0.0979             0.89                 0.7314          -0.1586          0.83              0.7333       -0.0967
D_ResNet            0.89        0.6964         -0.1936             0.90                 0.7043          -0.1957          0.89              0.6944       -0.1956

  'Paper ...' columns are published reference values.
  'Our ...' columns come from executed training runs.


---
## 4. Paper vs our accuracy graph

In [5]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
x = np.arange(len(paper_vs_ours))
w = 0.36
ax.bar(x - w / 2, paper_vs_ours["Paper Accuracy"], w, label="Paper (published)", color="#DD8452")
ax.bar(x + w / 2, paper_vs_ours["Our Accuracy"], w, label="Our reproduction", color="#4C72B0")
for i, (p, o) in enumerate(zip(paper_vs_ours["Paper Accuracy"], paper_vs_ours["Our Accuracy"])):
    ax.text(i - w / 2, p + 0.015, f"{p:.2f}", ha="center", fontsize=9)
    ax.text(i + w / 2, o + 0.015, f"{o:.3f}", ha="center", fontsize=9)
    ax.text(i, max(p, o) + 0.07, f"delta {o - p:+.3f}", ha="center", fontsize=8, color="#555555")
ax.set_xticks(x); ax.set_xticklabels(paper_vs_ours["Model"], rotation=15, ha="right")
ax.set_ylim(0, 1.12); ax.set_ylabel("test accuracy")
ax.set_title("Paper-reported accuracy vs our reproduced accuracy")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(R("comparisons") / "paper_vs_our_accuracy.png", dpi=150); plt.close()
print("  wrote results/comparisons/paper_vs_our_accuracy.png")

  wrote results/comparisons/paper_vs_our_accuracy.png


---
## 5. Final discussion

Generated from the executed numbers — nothing below is pre-written prose that
could contradict the actual results.

In [6]:
best_row = comparison.loc[comparison["Test Accuracy"].idxmax()]
worst_row = comparison.loc[comparison["Test Accuracy"].idxmin()]
mean_acc = comparison["Test Accuracy"].mean()
mean_delta = paper_vs_ours["Delta Accuracy"].mean() if len(paper_vs_ours) else np.nan

hist_path = R("metrics") / f"{best_row['Model']}_history.csv"
hist_note = ""
if hist_path.exists():
    h = pd.read_csv(hist_path)
    hist_note = (f"The best model, {best_row['Model']}, reached its peak validation accuracy at "
                 f"epoch {int(best_row['Best Epoch'])} and finished epoch {int(h['epoch'].max())}; "
                 f"its final training accuracy was {h['train_accuracy'].iloc[-1]:.4f} against a final "
                 f"validation accuracy of {h['val_accuracy'].iloc[-1]:.4f}.")

lines = []
lines.append("# Final Discussion\n")
lines.append("## 1. What was run\n")
lines.append(f"- Models trained: {', '.join(comparison['Model'])}")
lines.append(f"- Identical configuration for every model: {int(final_results['epochs'].iloc[0])} epochs, "
             f"batch size {int(final_results['batch_size'].iloc[0])}, "
             f"{final_results['optimizer'].iloc[0]} at lr {final_results['learning_rate'].iloc[0]}, "
             f"{final_results['loss'].iloc[0]}, seed {int(final_results['seed'].iloc[0])}.")
lines.append(f"- Best model by test accuracy: **{best_row['Model']}** "
             f"(test accuracy {best_row['Test Accuracy']:.4f}).")
lines.append(f"- Weakest model by test accuracy: {worst_row['Model']} "
             f"(test accuracy {worst_row['Test Accuracy']:.4f}).")
lines.append(f"- Mean test accuracy across the four models: {mean_acc:.4f}.")
lines.append(f"- Mean accuracy difference vs the paper's reported values: {mean_delta:+.4f} "
             "(our value minus paper value).\n")

lines.append("## 2. Reproduction limitations\n")
lines.append("**D-ResNet is an approximation.** The paper's D-ResNet is a 50-layer bottleneck residual "
             "network. Its original source implementation and training hyperparameters were not "
             "available, so `torchvision` ResNet50 was used as a practical approximation. The "
             "`D_ResNet` row in every table is ResNet50, **not** the original D-ResNet, and no claim "
             "of exact reproduction is made.\n")

lines.append("**Augmented dataset size differs from the paper.** The paper reports an augmented dataset "
             "of approximately 1468 images. This implementation applies the paper's described "
             "transforms (horizontal flip plus rotations of 0/30/45/90/180/270 degrees) to the "
             "training split only, producing a different count. The count was not forced to match. "
             "See `results/augmentation/augmentation_count_comparison.csv`.\n")

lines.append("**Dataset size differs from the paper's stated total.** The paper's text states 357 "
             "images, but the paper's own six class counts sum to 367, which is the number actually "
             "present in the dataset. Actual counts are reported in "
             "`results/dataset_summary/dataset_summary.csv`.\n")

lines.append("**Split ordering differs from the paper, deliberately.** The paper augments before "
             "splitting, which places rotated or flipped siblings of a training image into the test "
             "set. This implementation splits first at the source-image level and augments the "
             "training split only, so test images have no counterpart in training. This is a "
             "reproducibility safeguard required by the project specification, and it is the most "
             "likely reason our numbers differ from the paper's.\n")

lines.append("## 3. Why our accuracy may be lower than the paper's\n")
lines.append("Three factors, in order of expected importance:\n")
lines.append("1. **Leakage-free test set.** Our test images are unseen originals. The paper's protocol "
             "places augmented variants of training images into its test set, which inflates measured "
             "accuracy. Comparing the two directly is not like-for-like.")
lines.append("2. **Small test set.** The 15% clean test split is a few dozen images, so a single image "
             "moves accuracy by a couple of percentage points. Differences of a few points between "
             "models should not be over-interpreted.")
lines.append("3. **The 367-image dataset is tiny for six-way classification.** With no hyperparameter "
             "tuning and a single seed, every reported figure is a point estimate without a "
             "run-to-run error bar.\n")

if hist_note:
    lines.append("## 4. Training behaviour\n")
    lines.append(hist_note + "\n")
    gaps = [int(final_results.loc[final_results["model"] == m, "best_epoch"].iloc[0]) for m in comparison["Model"]]
    lines.append(f"Best validation epochs across models: {gaps}. "
                 "Earlier peaks indicate overfitting to the augmented training set rather than "
                 "incomplete fitting.\n")

lines.append("## 5. Metric definitions\n")
lines.append("- **Recall and sensitivity** are the same quantity for single-label multiclass "
             "classification and are both reported.")
lines.append("- **Specificity** is computed one-vs-rest as TN / (TN + FP) for each class, then "
             "macro- and weighted-averaged. sklearn does not provide this directly, so it is derived "
             "from the confusion matrix in notebook 02.")
lines.append("- Best-checkpoint selection used **validation accuracy only**. The test set was "
             "evaluated once per model, after training finished.\n")

lines.append("## 6. Not reproduced\n")
lines.append("- **Unseen 18-image experiment (3 images per class).** Not reproduced: no separate "
             "unseen dataset was available. Not fabricated.")
lines.append("- **RMSE / error analysis over SVM, CART, KNN, Random Forest, SGB, Cubist.** Not "
             "reproduced: the required data and methodology for those stages were unavailable. Not "
             "fabricated.\n")

lines.append("## 7. Artefact index\n")
lines.append("| Artefact | Path |")
lines.append("|---|---|")
lines.append("| Model comparison table | `results/comparisons/model_comparison.csv` |")
lines.append("| Model comparison graph | `results/comparisons/model_comparison.png` |")
lines.append("| Paper vs our results | `results/comparisons/paper_vs_our_results.csv` |")
lines.append("| Paper vs our accuracy | `results/comparisons/paper_vs_our_accuracy.png` |")
lines.append("| Per-model histories | `results/metrics/<Model>_history.csv` |")
lines.append("| Per-model reports | `results/metrics/<Model>_classification_report.csv` |")
lines.append("| Confusion matrices | `results/confusion_matrices/<Model>_confusion_matrix.{png,csv}` |")
lines.append("| Training curves | `results/training_curves/<Model>_{loss,accuracy}.png` |")
lines.append("| Checkpoints | Google Drive `BSM_Mineral_Classification/models/` |")

discussion = "\n".join(lines)
(R("comparisons") / "final_discussion.md").write_text(discussion, encoding="utf-8")
print(discussion)

# Final Discussion

## 1. What was run

- Models trained: VGG16, ResNet18, ResNet34, D_ResNet
- Identical configuration for every model: 60 epochs, batch size 32, Adam at lr 0.0001, CrossEntropyLoss, seed 42.
- Best model by test accuracy: **ResNet18** (test accuracy 0.8036).
- Weakest model by test accuracy: D_ResNet (test accuracy 0.6964).
- Mean test accuracy across the four models: 0.7500.
- Mean accuracy difference vs the paper's reported values: -0.0825 (our value minus paper value).

## 2. Reproduction limitations

**D-ResNet is an approximation.** The paper's D-ResNet is a 50-layer bottleneck residual network. Its original source implementation and training hyperparameters were not available, so `torchvision` ResNet50 was used as a practical approximation. The `D_ResNet` row in every table is ResNet50, **not** the original D-ResNet, and no claim of exact reproduction is made.

**Augmented dataset size differs from the paper.** The paper reports an augmented dataset of approxima

---
## Notebook 03 complete

| Artefact | Path |
|---|---|
| model comparison table | `results/comparisons/model_comparison.csv` |
| model comparison graph | `results/comparisons/model_comparison.png` |
| paper vs our results | `results/comparisons/paper_vs_our_results.csv` |
| paper vs our accuracy | `results/comparisons/paper_vs_our_accuracy.png` |
| final discussion | `results/comparisons/final_discussion.md` |

In [7]:
banner_lines = []
print("\n".join([
    "=" * 72,
    "  REQUIRED OUTPUT VERIFICATION",
    "=" * 72,
]))

required_outputs = [
    "dataset_summary/raw_dataset_inventory.csv",
    "dataset_summary/dataset_summary.csv",
    "dataset_summary/class_distribution.png",
    "dataset_summary/sample_images.png",
    "dataset_summary/image_dimensions.png",
    "preprocessing/data_wrangling_report.csv",
    "preprocessing/clean_dataset_manifest.csv",
    "preprocessing/data_imputation_report.csv",
    "preprocessing/final_split_manifest.csv",
    "preprocessing/dataset_version_summary.csv",
    "augmentation/augmentation_manifest.csv",
    "augmentation/augmentation_summary.csv",
    "augmentation/augmentation_examples.png",
    "metrics/final_results.csv",
    "metrics/training_time.csv",
    "comparisons/model_comparison.csv",
    "comparisons/model_comparison.png",
    "comparisons/paper_vs_our_results.csv",
    "comparisons/paper_vs_our_accuracy.png",
    "comparisons/final_discussion.md",
] + [f"metrics/{m}_history.csv" for m in MODEL_ORDER] \
  + [f"metrics/{m}_classification_report.csv" for m in MODEL_ORDER] \
  + [f"confusion_matrices/{m}_confusion_matrix.png" for m in MODEL_ORDER] \
  + [f"confusion_matrices/{m}_confusion_matrix.csv" for m in MODEL_ORDER] \
  + [f"training_curves/{m}_loss.png" for m in MODEL_ORDER] \
  + [f"training_curves/{m}_accuracy.png" for m in MODEL_ORDER]

missing = []
for rel in required_outputs:
    p = RESULTS / rel
    ok = p.exists() and p.stat().st_size > 0
    if not ok:
        missing.append(rel)
    print(f"  [{'OK ' if ok else 'MISS'}] results/{rel}")

print(f"\n  {len(required_outputs) - len(missing)}/{len(required_outputs)} required outputs present")
if missing:
    print("  MISSING:")
    for m in missing:
        print("   -", m)
else:
    print("  ALL REQUIRED OUTPUTS PRESENT")

  REQUIRED OUTPUT VERIFICATION
  [OK ] results/dataset_summary/raw_dataset_inventory.csv
  [OK ] results/dataset_summary/dataset_summary.csv
  [OK ] results/dataset_summary/class_distribution.png
  [OK ] results/dataset_summary/sample_images.png
  [OK ] results/dataset_summary/image_dimensions.png
  [OK ] results/preprocessing/data_wrangling_report.csv
  [OK ] results/preprocessing/clean_dataset_manifest.csv
  [OK ] results/preprocessing/data_imputation_report.csv
  [OK ] results/preprocessing/final_split_manifest.csv
  [OK ] results/preprocessing/dataset_version_summary.csv
  [OK ] results/augmentation/augmentation_manifest.csv
  [OK ] results/augmentation/augmentation_summary.csv
  [OK ] results/augmentation/augmentation_examples.png
  [OK ] results/metrics/final_results.csv
  [OK ] results/metrics/training_time.csv
  [OK ] results/comparisons/model_comparison.csv
  [OK ] results/comparisons/model_comparison.png
  [OK ] results/comparisons/paper_vs_our_results.csv
  [OK ] results/com


## Step 10 — Package results for hand-off

The full `results/` tree is zipped into a **single** file on Google Drive so the
results can be moved off the runtime without the Colab session being alive.

Checkpoints (`.pth`) are deliberately **excluded** — they are large and are
already stored separately under `BSM_Mineral_Classification/models/`. What lands
in GitHub is the lightweight evidence: CSVs, figures, and the **executed**
notebooks 01 and 02 (with their outputs), so the delivered code carries its own
proof of having run.

Notebook 03 cannot bundle itself, because `nbconvert --inplace` only writes a
notebook to disk once that notebook has finished executing. The cell below prints
the single command that appends it afterwards.


In [8]:

# ------------------------------------------------------------------
# Step 10.1 - zip results/, exclude checkpoints, mirror to Drive
# ------------------------------------------------------------------
import zipfile
import shutil
import json as _json
import time as _time

EXCLUDE_SUFFIXES = {".pth", ".pt", ".ckpt", ".h5", ".onnx"}
BUNDLE_DIR = Path("/content/drive/MyDrive/BSM_Mineral_Classification")
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
STAMP = _time.strftime("%Y%m%d-%H%M%S")
BUNDLE = BUNDLE_DIR / f"BSM_results_{STAMP}.zip"

packed, skipped = [], []
for p in sorted(RESULTS.rglob("*")):
    if not p.is_file():
        continue
    if p.suffix.lower() in EXCLUDE_SUFFIXES:
        skipped.append(str(p.relative_to(RESULTS)))
        continue
    packed.append(p)

# nbconvert --inplace writes each notebook to disk only AFTER that notebook has
# finished executing, so notebooks 01 and 02 are on disk in their EXECUTED form
# (with outputs) right now. Notebook 03 cannot include itself for the same
# reason - it is appended by one follow-up command once this run completes.
CURRENT_NB = Path("03_Final_Comparison.ipynb")
notebooks_packed = []
for nb in sorted((PROJECT_ROOT / "notebooks").glob("*.ipynb")):
    if nb.name == CURRENT_NB.name:
        continue
    packed.append(nb)
    notebooks_packed.append(f"notebooks/{nb.name}")

with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in packed:
        if p.suffix.lower() == ".ipynb":
            zf.write(p, arcname=f"notebooks/{p.name}")
        else:
            zf.write(p, arcname=str(p.relative_to(RESULTS)))

size_mb = BUNDLE.stat().st_size / 1e6
print(f"  packed {len(packed)} files ({len(notebooks_packed)} executed notebooks), "
      f"skipped {len(skipped)} checkpoint files")
print(f"  bundle: {BUNDLE}  ({size_mb:.2f} MB)")
if notebooks_packed:
    print("  executed notebooks in bundle:")
    for n in notebooks_packed:
        print("   -", n)
if skipped:
    print("  excluded (kept on Drive only):")
    for s in skipped:
        print("   -", s)

manifest = {
    "created_utc": _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
    "bundle": BUNDLE.name,
    "bundle_bytes": BUNDLE.stat().st_size,
    "file_count": len(packed),
    "executed_notebooks": notebooks_packed,
    "notebook_03_appended_after_run": True,
    "excluded_checkpoints": skipped,
    "files": sorted([str(p.relative_to(RESULTS)) for p in packed
                     if p.suffix.lower() != ".ipynb"] + notebooks_packed),
}
(BUNDLE_DIR / "results_manifest.json").write_text(_json.dumps(manifest, indent=2))

# keep a copy inside the runtime so the Colab Files panel can serve it too
try:
    shutil.copy2(BUNDLE, PROJECT_ROOT / BUNDLE.name)
    print(f"  runtime copy: {PROJECT_ROOT / BUNDLE.name}")
except Exception as e:
    print("  runtime copy skipped:", e)

print("\n  Download this ONE file from Google Drive and hand it over:")
print(f"    {BUNDLE}")
print("\n  Then append this notebook (it only exists on disk once the run ends):")
print("    cd /content/BSM-Project && python -c \"import zipfile,glob,os;\\")
print("    b=sorted(glob.glob('/content/drive/MyDrive/BSM_Mineral_Classification/"
      "BSM_results_*.zip'),key=os.path.getmtime)[-1];\\")
print("    z=zipfile.ZipFile(b,'a',zipfile.ZIP_DEFLATED);\\")
print("    n='notebooks/03_Final_Comparison.ipynb';\\")
print("    z.write(n,n); z.close(); print('appended',n,'to',b)\"")


  packed 60 files (2 executed notebooks), skipped 0 checkpoint files
  bundle: /content/drive/MyDrive/BSM_Mineral_Classification/BSM_results_20260930-215244.zip  (5.34 MB)
  executed notebooks in bundle:
   - notebooks/01_Data_Preprocessing_and_EDA.ipynb
   - notebooks/02_Model_Experiments.ipynb
  runtime copy: /content/BSM-Project/BSM_results_20260930-215244.zip

  Download this ONE file from Google Drive and hand it over:
    /content/drive/MyDrive/BSM_Mineral_Classification/BSM_results_20260930-215244.zip

  Then append this notebook (it only exists on disk once the run ends):
    cd /content/BSM-Project && python -c "import zipfile,glob,os;\
    b=sorted(glob.glob('/content/drive/MyDrive/BSM_Mineral_Classification/BSM_results_*.zip'),key=os.path.getmtime)[-1];\
    z=zipfile.ZipFile(b,'a',zipfile.ZIP_DEFLATED);\
    n='notebooks/03_Final_Comparison.ipynb';\
    z.write(n,n); z.close(); print('appended',n,'to',b)"



### What to do with the bundle

1. Download `BSM_Results_<timestamp>.zip` from
   `MyDrive/BSM_Mineral_Classification/`.
2. Run the printed `python -c` command once to append this notebook to the zip.
3. Unzip it over the repository's `results/` directory — `results/` for the data
   and `notebooks/` for the executed notebooks.
4. The bundle is re-checked before it is committed, and the repository's
   `.gitignore` still blocks `*.pth` / `*.pt` / `*.ckpt` as a second line of
   defence.
